# Asistente Bancario en Español — Preparación del Dataset

## GenAI Lifecycle: Definición

**Caso de uso:** Chatbot de atención al cliente para banca minorista en español.

**Problema:** Los modelos comerciales (GPT-4, Claude) son costosos para atención al cliente 24/7 y requieren enviar datos bancarios sensibles a terceros. Necesitamos evaluar si un modelo open-source pequeño puede ser una alternativa viable.

**Alcance:** Fine-tunear un modelo de 4B parámetros con ~5000 pares de instrucción/respuesta bancarios traducidos del inglés al español.

**Coste:** $0 (Google Colab Free + modelos open-source).

In [1]:
!pip install -qU datasets transformers sentencepiece sacremoses

from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from huggingface_hub import notebook_login
import pandas as pd

notebook_login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.8 MB/s eta 0:00:00


## 2. Dataset original

Elegí el dataset de Bitext porque es específico de banca minorista y ya viene estructurado en pares instrucción/respuesta, que es justo el formato que necesito para fine-tuning. El problema es que está en inglés, así que voy a tener que traducirlo.

In [2]:
dataset = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset")
print(f"Total samples: {len(dataset['train'])}")
print(f"Columnas: {dataset['train'].column_names}")
print(f"Categorías: {dataset['train'].unique('category')}")
print(f"Intents: {len(dataset['train'].unique('intent'))}")

# Mostrar ejemplos
dataset['train'].to_pandas().head(10)

README.md: 0.00B [00:00, ?B/s]

bitext-retail-banking-llm-chatbot-traini(…):   0%|          | 0.00/7.87M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25545 [00:00<?, ? examples/s]

Total samples: 25545
Columnas: ['tags', 'instruction', 'category', 'intent', 'response']
Categorías: ['CARD', 'LOAN', 'TRANSFER', 'FEES', 'ACCOUNT', 'CONTACT', 'ATM', 'FIND', 'PASSWORD']
Intents: 26


,tags,instruction,category,intent,response
0,BCIPZ,"I would like to acivate a card, can you help me?",CARD,activate_card,I'm here to assist you with that! Activating y...
1,BCILZ,"I have to activate an Visa online, how can I d...",CARD,activate_card,I'm here to assist you with activating your {{...
2,BCIPQZ,I'd like to actiate a card where do i do it,CARD,activate_card,I can help you with that! Activating your card...
3,BCLPQZ,I'd likke to activate a visa on mobile i need ...,CARD,activate_card,I'm here to assist you with activating your {{...
4,BCILPZ,"I would ilke to activate a credit card online,...",CARD,activate_card,I'm here to assist you with activating your cr...
5,BLZ,activate a cfredit card,CARD,activate_card,I'm here to help you activate your credit card...
6,BCIPZ,I received a new card and I want to activate i...,CARD,activate_card,Congratulations on receiving your new card! I'...
7,BLPQ,id like to activate a visa online,CARD,activate_card,I'm here to assist you with activating your vi...
8,BCIPZ,"I got to activaste a card, could I get some help?",CARD,activate_card,I'm here to assist you with activating your ca...
9,BCIPZ,"I have to activatean Amex on mobile, will you ...",CARD,activate_card,I'm here to assist you with activating your Am...


In [3]:
df = dataset['train'].to_pandas()

# Distribución por categoría
print("Distribución por categoría:")
print(df['category'].value_counts())

# Seleccionar 5000 samples balanceados
sampled = df.groupby('category', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 5000 // df['category'].nunique()), random_state=42)
)
print(f"\nSamples seleccionados: {len(sampled)}")

Distribución por categoría:
category
CARD        5980
LOAN        5954
ACCOUNT     2994
FIND        1998
CONTACT     1997
TRANSFER    1992
ATM         1983
PASSWORD    1700
FEES         947
Name: count, dtype: int64

Samples seleccionados: 4995


/tmp/ipython-input-2162122021.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = df.groupby('category', group_keys=False).apply(


## 3. Traducción EN → ES

No encontré un dataset bancario bueno directamente en español, así que decidí traducir con Helsinki-NLP/opus-mt-en-es. Es un modelo de traducción bastante liviano que corre bien en GPU. Traduzco en batches de 32 porque si mando todo junto se queda sin memoria.

In [4]:
import torch, gc

model_name = "Helsinki-NLP/opus-mt-en-es"
tokenizer_tr = AutoTokenizer.from_pretrained(model_name)
model_tr = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model_tr = model_tr.to("cuda")

def translate_batch(texts, batch_size=32):
    """Traduce una lista de textos en batches de 32 (seguro para T4 con textos largos)."""
    translations = []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer_tr(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            generated = model_tr.generate(**encoded)
        decoded = tokenizer_tr.batch_decode(generated, skip_special_tokens=True)
        translations.extend(decoded)
        print(f"  Traducidos {min(i+batch_size, total)}/{total}")
        # Liberar memoria GPU entre batches
        del encoded, generated
        torch.cuda.empty_cache()
    return translations

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [5]:
import os

SAVE_PATH = "traduccion_progreso.csv"

# Si ya existe el CSV con traducciones completas, cargar y no volver a traducir
if os.path.exists(SAVE_PATH):
    print("Cargando traducciones guardadas desde disco...")
    progress_df = pd.read_csv(SAVE_PATH)
    instructions_es = progress_df['instruction_es'].tolist()
    responses_es = progress_df['response_es'].tolist()
    categories = progress_df['category'].tolist()
    intents = progress_df['intent'].tolist()
    print(f"Cargadas {len(instructions_es)} instrucciones y {len(responses_es)} respuestas")
else:
    instructions_en = sampled['instruction'].tolist()
    responses_en = sampled['response'].tolist()
    categories = sampled['category'].tolist()
    intents = sampled['intent'].tolist()

    # Traducir instrucciones (textos cortos, van rápido)
    print("Traduciendo instrucciones...")
    instructions_es = translate_batch(instructions_en)
    print(f"Instrucciones traducidas: {len(instructions_es)}")

    # Liberar memoria antes de traducir respuestas (son más largas)
    gc.collect()
    torch.cuda.empty_cache()

    # Traducir respuestas (textos más largos, batch_size menor por seguridad)
    print("\nTraduciendo respuestas...")
    responses_es = translate_batch(responses_en, batch_size=16)
    print(f"Respuestas traducidas: {len(responses_es)}")

    # Guardar a disco para no perder progreso si el kernel se reinicia
    progress_df = pd.DataFrame({
        'instruction_es': instructions_es,
        'response_es': responses_es,
        'category': categories,
        'intent': intents,
    })
    progress_df.to_csv(SAVE_PATH, index=False)
    print(f"\nTraducciones guardadas en {SAVE_PATH}")

# Verificar ejemplos
for i in range(3):
    print(f"\n--- Ejemplo {i+1} ---")
    print(f"ES instruction: {instructions_es[i]}")
    print(f"ES response: {responses_es[i][:100]}...")

Traduciendo instrucciones...
  Traducidos 32/4995
  Traducidos 64/4995
  Traducidos 96/4995
  Traducidos 128/4995
  Traducidos 160/4995
  Traducidos 192/4995
  Traducidos 224/4995
  Traducidos 256/4995
  Traducidos 288/4995
  Traducidos 320/4995
  Traducidos 352/4995
  Traducidos 384/4995
  Traducidos 416/4995
  Traducidos 448/4995
  Traducidos 480/4995
  Traducidos 512/4995
  Traducidos 544/4995
  Traducidos 576/4995
  Traducidos 608/4995
  Traducidos 640/4995
  Traducidos 672/4995
  Traducidos 704/4995
  Traducidos 736/4995
  Traducidos 768/4995
  Traducidos 800/4995
  Traducidos 832/4995
  Traducidos 864/4995
  Traducidos 896/4995
  Traducidos 928/4995
  Traducidos 960/4995
  Traducidos 992/4995
  Traducidos 1024/4995
  Traducidos 1056/4995
  Traducidos 1088/4995
  Traducidos 1120/4995
  Traducidos 1152/4995
  Traducidos 1184/4995
  Traducidos 1216/4995
  Traducidos 1248/4995
  Traducidos 1280/4995
  Traducidos 1312/4995
  Traducidos 1344/4995
  Traducidos 1376/4995
  Traducidos 140

## 4. Armar el dataset final

Hago un split 90/10 para tener datos de test después. Conservo las columnas `category` e `intent` del dataset original porque me pueden servir para analizar en qué categorías el modelo anda mejor o peor.

In [6]:
new_df = pd.DataFrame({
    'instruction': instructions_es,
    'response': responses_es,
    'category': categories,
    'intent': intents,
})

full_dataset = Dataset.from_pandas(new_df)

# Split into train and test
dataset_dict = full_dataset.train_test_split(test_size=0.1, seed=42)

print(f"Train: {len(dataset_dict['train'])}, Test: {len(dataset_dict['test'])}")

Train: 4495, Test: 500


In [35]:
HF_USERNAME = "testlegadoss"
dataset_dict.push_to_hub(f"{HF_USERNAME}/retail-banking-chatbot-es", token=api.token)
print(f"Dataset subido a: https://huggingface.co/datasets/{HF_USERNAME}/retail-banking-chatbot-es")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 1.73MB / 1.73MB            

...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  198kB /  198kB            

README.md:   0%|          | 0.00/487 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Dataset subido a: https://huggingface.co/datasets/testlegadoss/retail-banking-chatbot-es


## Conclusiones — Fase Definición

- Dataset original: bitext/Bitext-retail-banking-llm-chatbot-training-dataset (inglés)
- Traducción: Helsinki-NLP/opus-mt-en-es
- Resultado: ~5000 pares instrucción/respuesta en español bancario
- Split: 90% train / 10% test
- Subido a HuggingFace Hub: [link]